## HuggingFaceTB/SmolLM2-1.7B-Instruct

In [1]:
import evaluate
import json
import os
import pandas as pd
import torch

from datasets import Dataset
from pathlib import Path
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

W0620 14:45:02.838000 31684 venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0620 14:45:02.877000 31684 venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


### DIRECTORIES

In [2]:
PROJECT_PATH = Path(os.getcwd()).parent.parent
DS_PATH = PROJECT_PATH / "datasets" / "fine-tuning" / "conversational"
TRAINING_DS_PATH = DS_PATH / "ordered_training_ds_juliet.jsonl"
TESTING_DS_PATH = DS_PATH / "ordered_testing_ds_juliet.jsonl"

LOGS_PATH = PROJECT_PATH / "logs" / "SmolLM2-1_7B-Instruct"
OUTPUT_METRICS_PATH = PROJECT_PATH / "output_metrics"
OUTPUT_METRICS_FILE = OUTPUT_METRICS_PATH / "SmolLM2-1_7B-Instruct.txt"

MODEL_OUTPUT_PATH = PROJECT_PATH / "fine-tuned-models" / "SmolLM2-1_7B-Instruct"

LOGS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_METRICS_PATH.mkdir(exist_ok=True)
MODEL_OUTPUT_PATH.mkdir(exist_ok=True)

### PARAMETERS

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

### TRAINING AND TESTING DATASET

In [4]:
train_data = []
with open(TRAINING_DS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line.strip()))

first_15K_training_examples = train_data[:15000]

### Refine dataset to Input - Label

In [5]:
training_dataframe = pd.DataFrame(first_15K_training_examples, columns=["messages", "cwe", "code", "filename", "type"])[["messages"]]

### Get Model and Tokenizer

In [6]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float8_e5m2,
    bnb_4bit_use_double_quant=True,
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    trust_remote_code=True,
    device_map="auto"
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [8]:
if tokenizer.chat_template is not None:
    print(tokenizer.chat_template)
else:
    print("No chat template")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [9]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (nor

### Tokenize Datasets

In [10]:
def tokenize_message(example):
    return tokenizer.apply_chat_template(example, tokenize=False)

training_dataset = Dataset.from_pandas(training_dataframe)

### Accuracy Compute

In [11]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    return {}

### LoRA Config

In [12]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [13]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

### TRAINING PARAMETERS

In [14]:
BATCH_SIZE = 2
LEARNING_RATE = 2e-5
LOGGING_STEPS = 200
GRADIENT_ACCUMULATION = 4
MAX_STEPS = 1875
WARMUP_RATIO = 0.1

In [15]:
training_args = SFTConfig(
    output_dir=MODEL_OUTPUT_PATH.resolve(),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    logging_dir=str(LOGS_PATH.resolve()),
    logging_steps=LOGGING_STEPS,
    max_steps=MAX_STEPS,
    fp16=True,
    learning_rate=LEARNING_RATE,
    save_total_limit=1,
    gradient_checkpointing=True,
    push_to_hub=False,
    eval_accumulation_steps=1,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [16]:
trainer = SFTTrainer(
    model=model,
    train_dataset=training_dataset,
    args=training_args,
    peft_config=peft_config,
)

C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\peft\tuners\lora\bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Tokenizing train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [17]:
trainer.train()

C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
200,1.670526
400,0.599473
600,0.340387
800,0.278042
1000,0.240859
1200,0.207842
1400,0.189876
1600,0.175667
1800,0.174617


C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences

TrainOutput(global_step=1875, training_loss=0.42018929341634115, metrics={'train_runtime': 8426.0955, 'train_samples_per_second': 1.78, 'train_steps_per_second': 0.223, 'total_flos': 1.18122172250112e+17, 'train_loss': 0.42018929341634115, 'entropy': 0.17286959879100322, 'num_tokens': 11023377.0, 'mean_token_accuracy': 0.9602654399474462, 'epoch': 1.0})

In [18]:
save_file_path = Path("~/ft/SmolLM2-1_7B-Instruct").expanduser()
trainer.model.save_pretrained(save_directory=save_file_path.resolve())